# W1 · Inspección de datos: horas extraordinarias en el personal a contrata del SSMC

**Propósito:** cargar, integrar y preparar la nómina de personal a contrata del Servicio de Salud Metropolitano Central (Transparencia Activa, código AO006) para inspeccionar las dos líneas candidatas del informe.

**Fuente:** Portal de Transparencia, 24 archivos mensuales (septiembre 2024 – agosto 2026), descargados el [fecha].

**Integrantes:** Alessandro Lavezzi - José Saavedra



In [1]:
import pandas as pd
import numpy as np
import seaborn as sbn
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

## 1. Lectura e integración de los archivos

Se leen los 24 archivos mensuales y se guardan en el diccionario `TA`, con nombres del tipo `TA_26_08` (año y mes).

Decisiones de lectura:
- **Separador `;` y codificación latin-1**, porque los archivos no están en UTF-8.
- **Todas las columnas como texto (`dtype=str`)**, porque varias mezclan formatos (por ejemplo, "Grado EUS o jornada" tiene "15" y "44 Hrs.", y los montos incluyen "$"). Así las conversiones se hacen después, de forma explícita.
- **Columna final vacía:** cada línea del archivo termina en `;`, lo que genera una columna sin nombre (`Unnamed: 21`). Se verifica que esté vacía y luego se elimina.
- **Columna `periodo`:** guarda el año-mes del archivo de origen, para identificar cada fila después de unir los meses.

In [ ]:
#LECTURA DE DATOS
DATA_DIR = Path("..") / "Data"
TA={}
for f in sorted(DATA_DIR.glob("TransparenciaActiva_*.csv")):
    fecha = f.stem.split("_")[-1]                  # "2026-08"
    nombre = f"TA_{fecha[2:4]}_{fecha[5:7]}"       # "TA_26_08"
    df = pd.read_csv(f, sep=";", encoding="latin-1",dtype=str)
    vacia = df["Unnamed: 21"].isna().all()         # la ultima columna esta vacia
    df = df.drop(columns=["Unnamed: 21"])          # se elimina
    df["periodo"] = fecha                          # se agrega columna año-mes del archivo
        
    
    globals()[nombre] = df
    TA[nombre] = df
    print(nombre, df.shape, "| columna vacía eliminada:", vacia)
    

TA_24_09 (1781, 22) | columna vacía eliminada: True
TA_24_10 (1799, 22) | columna vacía eliminada: True
TA_24_11 (1798, 22) | columna vacía eliminada: True
TA_24_12 (1803, 22) | columna vacía eliminada: True
TA_25_01 (1825, 22) | columna vacía eliminada: True
TA_25_02 (1842, 22) | columna vacía eliminada: True
TA_25_03 (1845, 22) | columna vacía eliminada: True
TA_25_04 (1861, 22) | columna vacía eliminada: True
TA_25_05 (1874, 22) | columna vacía eliminada: True
TA_25_06 (1870, 22) | columna vacía eliminada: True
TA_25_07 (1872, 22) | columna vacía eliminada: True
TA_25_08 (1884, 22) | columna vacía eliminada: True
TA_25_09 (1878, 22) | columna vacía eliminada: True
TA_25_10 (1882, 22) | columna vacía eliminada: True
TA_25_11 (1919, 22) | columna vacía eliminada: True
TA_25_12 (1918, 22) | columna vacía eliminada: True
TA_26_01 (1907, 22) | columna vacía eliminada: True
TA_26_02 (1951, 22) | columna vacía eliminada: True
TA_26_03 (1908, 22) | columna vacía eliminada: True
TA_26_04 (19

### 1.1 Unión de los 24 meses

Se unen los 24 archivos en una sola tabla (`df`). Para comprobar que no se perdieron ni duplicaron filas, se compara el número de filas de la tabla unida con la suma de las filas de los archivos individuales.

In [9]:
df=pd.concat(TA.values(), ignore_index=True)
suma_filas = sum(len(t) for t in TA.values())
print("Filas de la tabla unida:", len(df))
print("Suma de filas de los 24 meses:", suma_filas)
print("Columnas:", df.shape[1])
df

Filas de la tabla unida: 44777
Suma de filas de los 24 meses: 44777
Columnas: 22


,Año,Mes,Estamento,Nombre completo,Cargo o función,Grado EUS o jornada,Calificación profesional o formación,Región,Asignaciones especiales del mes (inc. en rem. bruta),"Remuneración bruta del mes (incluye bonos e incentivos, asig. especiales, horas extras)",Remuneración líquida del mes,Rem. adicionales del mes (no inc. en rem. bruta),Remuneración Bonos incentivos del mes (inc. en rem. bruta),Derecho a horas extraordinarias,Montos y horas extraordinarias diurnas del mes(inc. en rem. bruta),Montos y horas extraordinarias nocturnas del mes(inc. en rem. bruta),Montos y horas extraordinarias festivas del mes (inc. en rem. bruta),Fecha de inicio dd/mm/aa,Fecha de término dd/mm/aa,Viáticos del mes (no inc. en rem. bruta),Observaciones,periodo
0,2024,Septiembre,Auxiliar,"ABAITUA PIZARRO, GABRIEL ANTONIO",Chofer,20,AUXILIAR,Región Metropolitana de Santiago,(01),$ 838.003,$ 716.877,$ 0,$ 0,Sí,"$ 87.086 : 24,00 hrs",No tiene,No tiene,01/01/2024,31/12/2024,No informa,DIRECCION DE SERVICIO - REMUNERACION +BONO MEN...,2024-09
1,2024,Septiembre,Técnico,"ABARCA MATURANA, ARMANDO IVAN",ATENCION CLINICA,22,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(01),$ 779.837,$ 564.066,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/09/2024,30/09/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
2,2024,Septiembre,Profesional,"ABARCA MUÑOZ, MARITZA DENISSE",APOYO ADMINISTRATIVO,9,ABOGADO (A),Región Metropolitana de Santiago,(01),$ 2.404.046,$ 1.869.045,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/01/2024,31/12/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
3,2024,Septiembre,Auxiliar,"ABARCA ROJAS, MANUEL EDUARDO",CONDUCTOR,24,AUXILIAR,Región Metropolitana de Santiago,(205)(203)(204)(186),$ 845.475,$ 655.884,$ 0,$ 94.920,No,No tiene,No tiene,No tiene,01/01/2024,31/12/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
4,2024,Septiembre,Técnico,"ABARCA RUBIO, FERNANDA DANIELA",ATENCION CLINICA,22,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(202)(204)(186),$ 1.040.110,$ 503.890,$ 0,$ 112.666,No,No tiene,No tiene,No tiene,01/01/2024,31/12/2024,No informa,CENTRO METROPOLITANO DE ATENCION PREHOSPITALAR...,2024-09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44772,2026,Agosto,Auxiliar,"ZUÑIGA FLORES, MARCELO PAUL",CONDUCTOR,21,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(205)(202)(204)(186),$ 1.587.591,$ 1.109.259,$ 0,$ 0,Sí,"$ 150.595 : 40,00 hrs","$ 189.749 : 42,00 hrs",No tiene,01/01/2026,31/12/2026,No informa,REMUNERACION AGO.2026,2026-08
44773,2026,Agosto,Profesional,"ZUÑIGA JARAMILLO, JONATHAN ROLANDO",Apoyo Administrativo,13,PSICOLOGO (A),Región Metropolitana de Santiago,(186),$ 1.908.175,$ 1.537.874,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/01/2026,31/12/2026,No informa,Sin observaciones,2026-08
44774,2026,Agosto,Técnico,"ZUÑIGA OROZCO, ROMINA JAZMIN",ATENCION CLINICA,22,TECNICO NIVEL SUPERIOR ENFERMERIA,Región Metropolitana de Santiago,(202)(204)(186),$ 1.092.182,$ 824.661,$ 0,$ 0,No,No tiene,No tiene,No tiene,01/01/2026,31/12/2026,No informa,REMUNERACION AGO.2026,2026-08
44775,2026,Agosto,Técnico,"ZUÑIGA RIQUELME, CLAUDIO ALBERTO",ATENCION CLINICA,22,AUXILIAR DE ENFERMERIA,Región Metropolitana de Santiago,(202)(204)(186),$ 1.316.038,$ 858.615,$ 0,$ 0,Sí,"$ 52.347 : 15,00 hrs","$ 180.073 : 43,00 hrs",No tiene,01/01/2026,31/12/2026,No informa,REMUNERACION AGO.2026,2026-08


### Resultado de la sección 1

- Se cargaron los 24 archivos mensuales (septiembre 2024 – agosto 2026).
- La columna final sin nombre estaba vacía en los 24 archivos y se eliminó.
- La tabla unida tiene **44.777 filas**, igual a la suma de las filas de los 24 archivos, por lo que la unión no perdió ni duplicó registros.
- La tabla tiene 22 columnas: las 21 originales con información más `periodo`.
- El número de registros por mes varía entre 1.781 (septiembre 2024) y 1.951 (febrero 2026).

**Qué no permite concluir todavía:** cada fila es un registro del archivo, no necesariamente una persona distinta. Si una persona tiene más de un contrato, aparece en más de una fila. Esto se revisa en la sección 2.